# Lahore Smog Forecast — 24h PM2.5 prediction with LSTM
Predicts the next 24 hours of PM2.5 in Lahore, Pakistan from the past 72 hours of pollution + weather data. Runs on Colab free tier (CPU is fine, GPU is faster).

In [ ]:
!pip install -q torch pandas numpy requests matplotlib

## 1. Fetch data
Hourly PM2.5/PM10 (CAMS) and weather (ERA5) for Lahore from the free, keyless Open-Meteo APIs.

In [ ]:
import requests, pandas as pd
LAT, LON = 31.5497, 74.3436
START, END = "2022-01-01", "2026-10-09"
aq = requests.get("https://air-quality-api.open-meteo.com/v1/air-quality",
    params={"latitude": LAT, "longitude": LON, "hourly": "pm2_5,pm10",
            "start_date": START, "end_date": END, "timezone": "auto"}, timeout=120).json()
wx = requests.get("https://archive-api.open-meteo.com/v1/archive",
    params={"latitude": LAT, "longitude": LON,
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure",
            "start_date": START, "end_date": END, "timezone": "auto"}, timeout=120).json()
df = pd.DataFrame({"time": pd.to_datetime(aq["hourly"]["time"])})
df["pm25"], df["pm10"] = aq["hourly"]["pm2_5"], aq["hourly"]["pm10"]
for k in ["temperature_2m","relative_humidity_2m","wind_speed_10m","surface_pressure"]:
    df[k] = wx["hourly"][k]
df = df.sort_values("time").interpolate(limit_direction="both").dropna().reset_index(drop=True)
print(len(df), "hourly rows")

## 2. Features
Cyclical time encodings so the model learns daily/seasonal smog cycles.

In [ ]:
import numpy as np
FEATS = ["pm25","pm10","temperature_2m","relative_humidity_2m","wind_speed_10m","surface_pressure"]
df["hour_sin"] = np.sin(2*np.pi*df.time.dt.hour/24)
df["hour_cos"] = np.cos(2*np.pi*df.time.dt.hour/24)
df["doy_sin"]  = np.sin(2*np.pi*df.time.dt.dayofyear/365)
df["doy_cos"]  = np.cos(2*np.pi*df.time.dt.dayofyear/365)
ALL = FEATS + ["hour_sin","hour_cos","doy_sin","doy_cos"]
LOOKBACK, HORIZON = 72, 24

## 3. Chronological split + windowing
Train 2022–23, validate 2024, test 2025–26. No shuffling across time — no leakage.

In [ ]:
X_all = df[ALL].to_numpy(np.float32); y_all = df["pm25"].to_numpy(np.float32)
def windows(X, y):
    n = len(X)-LOOKBACK-HORIZON+1
    return (np.stack([X[i:i+LOOKBACK] for i in range(n)]),
            np.stack([y[i+LOOKBACK:i+LOOKBACK+HORIZON] for i in range(n)]))
def split(mask):
    i = np.where(mask)[0]; return windows(X_all[i.min():i.max()+1], y_all[i.min():i.max()+1])
Xtr,ytr = split(df.time < "2024-01-01")
Xva,yva = split((df.time>="2024-01-01")&(df.time<"2025-01-01"))
Xte,yte = split(df.time >= "2025-01-01")
mu, sd = Xtr.reshape(-1,len(ALL)).mean(0), Xtr.reshape(-1,len(ALL)).std(0)+1e-8
ymu, ysd = ytr.mean(), ytr.std()+1e-8
Xtr,Xva,Xte = (Xtr-mu)/sd, (Xva-mu)/sd, (Xte-mu)/sd
ytr_n, yva_n = (ytr-ymu)/ysd, (yva-ymu)/ysd
print("train/val/test windows:", len(Xtr), len(Xva), len(Xte))

## 4. Model — 2-layer LSTM

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
class SmogLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(len(ALL), 64, 2, batch_first=True, dropout=0.2)
        self.head = nn.Linear(64, HORIZON)
    def forward(self, x):
        _, (h, _) = self.lstm(x); return self.head(h[-1])
torch.manual_seed(0)
model = SmogLSTM()
opt = torch.optim.Adam(model.parameters(), lr=1e-3); loss_fn = nn.MSELoss()
dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr_n)),
                batch_size=256, shuffle=True)
Xva_t, yva_t = torch.from_numpy(Xva), torch.from_numpy(yva_n)
best, pat = float("inf"), 0
for ep in range(60):
    model.train()
    for xb, yb in dl:
        opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
    model.eval()
    with torch.no_grad(): vl = loss_fn(model(Xva_t), yva_t).item()
    print(f"epoch {ep+1}: val_mse={vl:.4f}")
    if vl < best-1e-4: best, pat = vl, 0; torch.save(model.state_dict(), "model.pt")
    else:
        pat += 1
        if pat >= 8: print("early stop"); break

## 5. Evaluate vs baselines
Persistence (repeat last value) and hourly climatology — the model must beat both.

In [ ]:
model.load_state_dict(torch.load("model.pt", weights_only=True)); model.eval()
with torch.no_grad():
    pred = model(torch.from_numpy(Xte)).numpy()*ysd + ymu
persist = np.repeat(yte[:, -1:], HORIZON, axis=1)
clim = np.tile(ytr.reshape(-1,HORIZON).mean(0), (len(yte),1))
def scores(p,t):
    e=p-t; return np.mean(np.abs(e)), np.sqrt(np.mean(e**2))
for name,p in [("LSTM",pred),("persistence",persist),("climatology",clim)]:
    mae,rmse = scores(p,yte); print(f"{name:12s} MAE={mae:6.2f}  RMSE={rmse:6.2f}")

## 6. Visual check — forecast vs actual on a smog episode

In [ ]:
import matplotlib.pyplot as plt
i = int(np.argmax(yte.max(1)))   # worst smog day in test set
plt.figure(figsize=(12,4)); plt.plot(yte[i], label="actual"); plt.plot(pred[i], label="LSTM forecast")
plt.legend(); plt.ylabel("PM2.5 µg/m³"); plt.title("24h forecast on worst test-set smog day"); plt.show()

## 7. Save artifacts
`model.pt` + normalization stats ship with the Streamlit app.

In [ ]:
import json
json.dump({"feat_mu":mu.tolist(),"feat_sd":sd.tolist(),"y_mu":float(ymu),"y_sd":float(ysd),
           "features":ALL,"lookback":LOOKBACK,"horizon":HORIZON}, open("scaler.json","w"))
print("saved model.pt, scaler.json")